# Stochastic Processes, White Noise, and Propagating Uncertainty

*Supplement to the **Kalman Filter Boot Camp**. Continues [04 · Random Variables](04_Random_Variables.ipynb) from single random vectors to random **processes** (signals) — and shows how a dynamic system moves a whole probability distribution around, which is exactly what the Kalman filter's covariance equations do.*

**Style:** every equation gets a plain-language paraphrase (→); extra intuition is flagged **→ Intuition**.

### 🧩 From Random Variables to Random Processes

- A **random (stochastic) process** $X(t)$ is a random variable *for every time $t$* — a whole family of RVs indexed by time. One "draw" is an entire signal (a **sample path**).

- It is described (to second order) by its mean function and its autocorrelation:

$$
\mu_X(t) = \mathbb{E}[X(t)], \qquad R_X(t_1,t_2) = \mathbb{E}\!\big[X(t_1)X(t_2)^T\big].
$$

  → The mean says where the signal sits on average at each instant; the autocorrelation says how the value at one instant relates to the value at another.

- The process is **wide-sense stationary (WSS)** if $\mu_X$ is constant and $R_X$ depends only on the lag $\tau = t_2-t_1$:

$$
R_X(\tau) = \mathbb{E}\!\big[X(t)X(t+\tau)^T\big].
$$

  → "Statistically the same at all times" — only the *gap* between two instants matters, not where they sit on the clock.

- **Autocovariance** removes the mean: $\;C_X(\tau) = R_X(\tau) - \mu_X\mu_X^T$. For zero-mean processes autocovariance = autocorrelation (same simplification as in [04](04_Random_Variables.ipynb)).

### 🧩 White Noise — the Idealized Driving Signal

- **White noise** $w(t)$ is a zero-mean process whose autocorrelation is an impulse in lag:

$$
\mathbb{E}[w(t)\,w(t+\tau)^T] = S_w\,\delta(\tau).
$$

  → Any two *distinct* instants are **uncorrelated**, no matter how close: the signal has no memory at all. $S_w$ is the **power spectral density (PSD)** — the strength per unit bandwidth.

- Its power spectrum (Fourier transform of $R_w$) is **flat**:

$$
\Phi_w(\omega) = S_w \quad \text{for all } \omega.
$$

  → "White" because, like white light, it contains **every frequency in equal measure**.

- **→ Intuition:** true white noise has infinite power and is a mathematical idealization, but it is the perfect *building block*: real, correlated noises are modeled as white noise passed through a filter (next). In the KF, $w_k$ and $v_k$ are assumed white so that today's noise carries no information the filter already used yesterday.

---

### 🧩 Discrete-Time White Noise

- In discrete time the impulse becomes a Kronecker delta:

$$
\mathbb{E}[w_k\,w_j^T] = \Sigma_{\tilde w}\,\delta_{kj} = \begin{cases}\Sigma_{\tilde w}, & k=j,\\[2pt] 0, & k\neq j.\end{cases}
$$

  → Every sample is uncorrelated with every other sample; $\Sigma_{\tilde w}$ is an ordinary (finite) covariance matrix — much friendlier than the continuous PSD.

### 🧩 Shaping Filters — Making Colored Noise from White Noise

- Real disturbances are usually **correlated in time** ("colored"). Model them as the output of a linear system driven by white noise — a **shaping filter**:

$$
\dot{x}_s(t) = A_s x_s(t) + B_s\, w(t), \qquad n(t) = C_s x_s(t),
$$

  with $w(t)$ white. A common scalar case is the **Gauss–Markov** (first-order) process:

$$
\dot{n}(t) = -\tfrac{1}{\tau_c}\,n(t) + w(t) \;\;\Longrightarrow\;\; R_n(\tau) = \sigma_n^2\, e^{-|\tau|/\tau_c}.
$$

  → The correlation decays exponentially with a **correlation time** $\tau_c$: samples closer than $\tau_c$ act alike, samples far apart look independent. Large $\tau_c$ = slowly-drifting noise; $\tau_c\to 0$ recovers white noise.

- **→ Intuition / KF use:** if a KF's noise is *not* white, you can often **augment the state** with the shaping-filter state $x_s$. Then the driving noise of the augmented system *is* white and the standard KF assumptions hold again — this is exactly the trick used later for [colored noise](09_Making_the_KF_Bulletproof.ipynb).

### 🧩 Functions of Random Variables & Simulating Correlated Gaussians

- Push a Gaussian through a linear map: if $x \sim \mathcal{N}(\bar{x},\Sigma_x)$ and $y = Fx + b$, then

$$
y \sim \mathcal{N}\big(F\bar{x}+b,\; F\Sigma_x F^T\big).
$$

  → A linear transform moves the mean the obvious way and **sandwiches** the covariance as $F\Sigma_x F^T$. Gaussianity is preserved — the deep reason the (linear) KF stays Gaussian forever.

- To *generate* a sample with a desired covariance $\Sigma$, factor it (Cholesky) as $\Sigma = LL^T$ and set

$$
x = \bar{x} + L\,\eta, \qquad \eta \sim \mathcal{N}(0, I).
$$

  → Draw plain independent unit-variance noise $\eta$, then "color" it with $L$; the result has exactly covariance $L\,I\,L^T = \Sigma$. (In Octave: `L = chol(Sigma,'lower'); x = xbar + L*randn(n,1);`)

### 🧩 Propagating Mean and Covariance Through a Dynamic System

- Take the stochastic discrete model $x_{k+1} = A_k x_k + B_k u_k + w_k$ with $w_k\sim(0,\Sigma_{\tilde w})$, $w_k$ white and independent of $x_k$.

- **Mean** propagates through the deterministic part:

$$
\bar{x}_{k+1} = A_k \bar{x}_k + B_k u_k.
$$

  → The average trajectory just obeys the noise-free dynamics — noise averages out of the mean.

- **Covariance** propagates by the discrete **Lyapunov recursion**:

$$
\Sigma_{x,k+1} = A_k\,\Sigma_{x,k}\,A_k^T + \Sigma_{\tilde w}.
$$

  → Uncertainty is **squeezed/rotated** by the dynamics ($A\Sigma A^T$) and then **grown** by fresh process noise ($+\Sigma_{\tilde w}$). This is *identical* to the Kalman filter's prediction-step covariance update — the KF prediction is just uncertainty propagation.

- Continuous-time analogue (the **Lyapunov ODE**):

$$
\dot{\Sigma}_x(t) = A\,\Sigma_x(t) + \Sigma_x(t)\,A^T + B_w S_w B_w^T.
$$

  → Same two effects in rate form: dynamics reshape the covariance while process noise continuously injects uncertainty.

### 🧩 Relating Continuous Noise to Discrete Noise (Van Loan)

- When we sample a continuous plant $\dot{x}=Ax+B_w w$ (with $\mathbb{E}[w w^T]=S_w\delta$) at period $\Delta t$, we need the *discrete* process-noise covariance $\Sigma_{\tilde w}$ that reproduces the true accumulation over one step:

$$
\Sigma_{\tilde w} = \int_{0}^{\Delta t} e^{A\eta}\,B_w S_w B_w^T\, e^{A^T\eta}\, d\eta .
$$

  → Integrate the continuous noise covariance, propagated by the STM, over one sample interval — the noise poured in **between** samples, collapsed into one equivalent kick.

- **Van Loan's trick** computes both $A_d$ and $\Sigma_{\tilde w}$ from a single matrix exponential. Form

$$
\mathcal{M} = \exp\!\left(\begin{bmatrix} -A & B_w S_w B_w^T \\ 0 & A^T\end{bmatrix}\Delta t\right) = \begin{bmatrix} \star & A_d^{-1}\Sigma_{\tilde w} \\ 0 & A_d^{T}\end{bmatrix},
$$

  then read off $A_d = (\text{lower-right block})^T$ and $\Sigma_{\tilde w} = A_d \cdot(\text{upper-right block})$.

  → One exponential of a cleverly stacked matrix hands you the discrete dynamics **and** the discrete noise covariance together — no fragile numerical integration.

### 🧩 Summary

- A **stochastic process** is a random variable at every time; to second order it is captured by its mean and **autocorrelation** $R_X(\tau)$; **WSS** means these don't drift with absolute time.

- **White noise**: uncorrelated across time, flat spectrum, $\mathbb{E}[w_k w_j^T]=\Sigma_{\tilde w}\delta_{kj}$ — the memoryless building block the KF assumes for $w_k, v_k$.

- **Shaping filters** turn white noise into realistic **colored** noise; state augmentation reverses the trick to restore KF assumptions.

- Linear maps keep Gaussians Gaussian and transform covariance as $F\Sigma F^T$; **Cholesky** lets you *simulate* any target covariance.

- Uncertainty propagates by the **Lyapunov** recursion $\Sigma_{k+1}=A\Sigma_k A^T + \Sigma_{\tilde w}$ — literally the KF prediction step — and continuous noise converts to discrete $\Sigma_{\tilde w}$ via the **Van Loan** matrix-exponential trick.

---
*Next: [07 · Sequential Probabilistic Inference & the Six Steps](07_Sequential_Probabilistic_Inference_Six_Steps.ipynb).*